In [6]:
!pip install optuna
import pandas as pd
import numpy as np
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# 1. DATA FILTERING (FIRST ROUND ONLY)
df = pd.read_csv('men_2026_matchups_training.csv')
df_first_round = df[df['round'] == 'Second Round'].copy()

# Features Pool (40 variables)
features_40 = [
    '5man_bpm',
    'torvik_rtg',
    '3man_bpm',
    'kenpom_rtg',
    'wab',
    '3man_obpm',
    '5man_obpm',
    'size_speed_index',
    '5man_dbpm',
    '5man_dprpg',
    '3man_dprpg',
    'experience_weighted_production',
    '3man_dbpm',
    'lineup_depth_quality',
    'def_dunk_share',
    'bench_scoring_ratio',
    'height',
    'size',
    'def_lineup_depth_quality',
    '5man_prpg',
    'def_size_speed_index',
    'def_experience_impact',
    'efg_pct',
    'rim_to_three_ratio',
    'def_dunk_fg_pct',
    '2p_pct',
    'free_throw_advantage',
    'bench',
    'tor',
    'astd_pct',
    'def_3pt_fg_pct',
    'tord',
    'offensive_versatility_score',
    '2pd_pct',
    'ast_pct',
    '3p_pct',
    '3man_prpg',
    'off_close2_fg_pct',
    'def_rim_to_three_ratio',
    'net_ftr_margin'
]

X = df_first_round[features_40].fillna(df_first_round[features_40].median())
y = df_first_round['win']

# 2. SPLITS & SCALING
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.18, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# 3. FEATURE SELECTION (TOP 25 FROM 40)
selector = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=42),
    max_features=25,
    threshold=-np.inf
)
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_val_sel = selector.transform(X_val_scaled)
selected_names = np.array(features_40)[selector.get_support()]

# 4. OPTIMIZATION STUDY
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 700),
        'max_depth': trial.suggest_int('max_depth', 5, 18),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 12),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 6),
        'ccp_alpha': trial.suggest_float('ccp_alpha', 1e-6, 5e-4, log=True),
        'max_features': 'sqrt',
        'random_state': 42,
        'n_jobs': -1
    }
    rf = RandomForestClassifier(**params)
    return cross_val_score(rf, X_train_sel, y_train, cv=5, n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# 5. OUTPUT BEST HYPERPARAMETERS
print("\n" + "="*30)
print("BEST HYPERPARAMETERS FOR FIRST ROUND MODEL")
print("="*30)
for key, value in study.best_params.items():
    print(f"{key}: {value}")
print("="*30)
print(f"Best CV Mean: {study.best_value:.4f}")

# Final Validation Check
best_rf = RandomForestClassifier(**study.best_params, random_state=42)
best_rf.fit(X_train_sel, y_train)
val_acc = accuracy_score(y_val, best_rf.predict(X_val_sel))
print(f"Validation Accuracy: {val_acc:.4f}")

[I 2026-03-19 01:43:16,139] A new study created in memory with name: no-name-eb748759-9f12-4129-b849-d9937200752a
[I 2026-03-19 01:43:30,842] Trial 0 finished with value: 0.7332323232323232 and parameters: {'n_estimators': 279, 'max_depth': 13, 'min_samples_split': 11, 'min_samples_leaf': 6, 'ccp_alpha': 1.3985554391775906e-05}. Best is trial 0 with value: 0.7332323232323232.
[I 2026-03-19 01:43:36,885] Trial 1 finished with value: 0.7422222222222222 and parameters: {'n_estimators': 235, 'max_depth': 6, 'min_samples_split': 10, 'min_samples_leaf': 5, 'ccp_alpha': 0.0003362978466663972}. Best is trial 1 with value: 0.7422222222222222.
[I 2026-03-19 01:43:43,018] Trial 2 finished with value: 0.7376767676767677 and parameters: {'n_estimators': 307, 'max_depth': 7, 'min_samples_split': 9, 'min_samples_leaf': 4, 'ccp_alpha': 5.548613386357346e-06}. Best is trial 1 with value: 0.7422222222222222.
[I 2026-03-19 01:43:58,042] Trial 3 finished with value: 0.7466666666666667 and parameters: {'n_


BEST HYPERPARAMETERS FOR FIRST ROUND MODEL
n_estimators: 696
max_depth: 17
min_samples_split: 12
min_samples_leaf: 2
ccp_alpha: 0.0003089191064117835
Best CV Mean: 0.7467
Validation Accuracy: 0.8367


In [8]:
# Print the 25 variables that survived the selection
selected_names = np.array(features_40)[selector.get_support()]
print("Variables used in the final model:")
print(selected_names)

Variables used in the final model:
['5man_bpm' 'torvik_rtg' '3man_bpm' 'kenpom_rtg' 'wab' '3man_obpm'
 '5man_obpm' '5man_dbpm' '5man_dprpg' '3man_dprpg'
 'experience_weighted_production' 'lineup_depth_quality' 'def_dunk_share'
 'size' '5man_prpg' 'def_experience_impact' 'efg_pct' 'def_dunk_fg_pct'
 '2p_pct' 'free_throw_advantage' 'tor' 'astd_pct' 'tord'
 'offensive_versatility_score' '2pd_pct']
